In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
import random
from collections import deque

# ================= 1. 自定义层定义 (必须在 load_model 前声明) =================

class TransformerBlock(Layer):
    """Transformer 编码器块"""
    def __init__(self, embed_dim=128, num_heads=8, ff_dim=256, rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.embed_dim, self.num_heads, self.ff_dim = embed_dim, num_heads, ff_dim
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([Dense(ff_dim, activation="gelu"), Dense(embed_dim)])
        self.layernorm1, self.layernorm2 = LayerNormalization(epsilon=1e-6), LayerNormalization(epsilon=1e-6)
        self.dropout1, self.dropout2 = Dropout(rate), Dropout(rate)

    def call(self, inputs, training=None):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

    def get_config(self):
        """支持 Keras 序列化"""
        config = super().get_config()
        config.update({"embed_dim": self.embed_dim, "num_heads": self.num_heads, "ff_dim": self.ff_dim})
        return config

# ================= 2. 卫星抗干扰环境仿真 =================

class SatelliteEnv:
    """基于态势真值的闭环验证环境"""
    def __init__(self, num_channels=10):
        self.num_channels = num_channels
        
    def step(self, action, true_occupancy):
        # 避障逻辑：如果选择信道对应真值为 1，则发生碰撞
        collision = (true_occupancy[action] == 1.0)
        # 奖励函数：避障成功 +1，碰撞 -2
        reward = 1.0 if not collision else -2.0
        return reward, collision

# ================= 3. DQN 决策 Agent =================

class DQNAgent:
    """深度 Q 网络代理"""
    def __init__(self, state_size=10, action_size=10):
        self.state_size, self.action_size = state_size, action_size
        self.memory = deque(maxlen=2000)
        self.gamma, self.epsilon = 0.95, 1.0
        self.epsilon_min, self.epsilon_decay = 0.01, 0.995
        self.model = self._build_model()

    def _build_model(self):
        model = Sequential([
            Dense(64, input_dim=self.state_size, activation='relu'),
            Dense(64, activation='relu'),
            Dense(self.action_size, activation='linear')
        ])
        model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
        return model

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        act_values = self.model.predict(state, verbose=0)
        return np.argmax(act_values[0])

# ================= 4. 级联推理与决策流水线 =================

def run_decision_pipeline():
    # 路径配置
    data_path = "/root/autodl-tmp/validate/0218/Prediction/situation_data_enhanced"
    model_path = "/root/autodl-tmp/validate/0218/Prediction/Transformer/models/best_hybrid_situation.h5"
    
    # 1. 加载测试数据
    print("🚀 载入态势测试集...")
    try:
        X_test = np.load(f"{data_path}/X_test.npy")
        y_test = np.load(f"{data_path}/y_test.npy")
    except Exception as e:
        print(f"❌ 数据加载失败: {e}"); return

    # 2. 核心修复：加载模型并传入自定义对象
    print(f"🧠 正在加载预测模型并解析 TransformerBlock...")
    if os.path.exists(model_path):
        # 显式映射自定义层，解决 ValueError
        custom_dict = {'TransformerBlock': TransformerBlock}
        prediction_model = tf.keras.models.load_model(model_path, custom_objects=custom_dict, compile=False)
        print("✅ 模型加载成功")
    else:
        print("❌ 找不到模型文件"); return

    # 3. 初始化环境与 Agent
    env = SatelliteEnv(num_channels=10)
    agent = DQNAgent(state_size=10, action_size=10)
    
    test_episodes = 100
    success_count = 0
    
    print(f"\n🔥 启动预测驱动的避障决策验证 (测试步数: {test_episodes})...")
    
    for i in range(test_episodes):
        # 随机抽取一个测试时隙
        idx = random.randint(0, len(X_test) - 1)
        current_input = X_test[idx:idx+1]
        
        # 步骤 A: 态势预测 (补偿星地时延)
        predicted_probs = prediction_model.predict(current_input, verbose=0)
        
        # 步骤 B: RL 智能决策
        action = agent.act(predicted_probs)
        
        # 步骤 C: 环境验证 (使用未来时隙真实占用情况)
        reward, collision = env.step(action, y_test[idx])
        
        if not collision: success_count += 1
        
        if (i+1) % 20 == 0:
            current_rate = (success_count / (i+1)) * 100
            print(f"📡 Step: {i+1:3d} | 当前避障成功率: {current_rate:6.2f}%")

    print("\n" + "="*45)
    print(f"🎯 最终系统评估完成")
    print(f"🏆 全流程避障成功率: {(success_count/test_episodes)*100:.2f}%")
    print("="*45)

if __name__ == "__main__":
    run_decision_pipeline()

2026-02-24 14:14:43.096876: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 14:14:43.154900: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 14:14:44.057261: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


🚀 载入态势测试集...
🧠 正在加载预测模型并解析 TransformerBlock...


2026-02-24 14:14:45.388589: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22182 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:27:00.0, compute capability: 8.9


✅ 模型加载成功

🔥 启动预测驱动的避障决策验证 (测试步数: 100)...


2026-02-24 14:14:46.820730: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:424] Loaded cuDNN version 8600
2026-02-24 14:14:47.095131: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:637] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


📡 Step:  20 | 当前避障成功率: 100.00%
📡 Step:  40 | 当前避障成功率:  92.50%
📡 Step:  60 | 当前避障成功率:  90.00%
📡 Step:  80 | 当前避障成功率:  91.25%
📡 Step: 100 | 当前避障成功率:  89.00%

🎯 最终系统评估完成
🏆 全流程避障成功率: 89.00%


In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
import random
from collections import deque

# ================= 1. 自定义层加载环境 (必须保留) =================
class TransformerBlock(Layer):
    def __init__(self, embed_dim=128, num_heads=8, ff_dim=256, rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.embed_dim, self.num_heads, self.ff_dim = embed_dim, num_heads, ff_dim
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([Dense(ff_dim, activation="gelu"), Dense(embed_dim)])
        self.layernorm1, self.layernorm2 = LayerNormalization(epsilon=1e-6), LayerNormalization(epsilon=1e-6)
        self.dropout1, self.dropout2 = Dropout(rate), Dropout(rate)

    def call(self, inputs, training=None):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + self.dropout1(attn_output, training=training))
        return self.layernorm2(out1 + self.dropout2(self.ffn(out1), training=training))

    def get_config(self):
        config = super().get_config()
        config.update({"embed_dim": self.embed_dim, "num_heads": self.num_heads, "ff_dim": self.ff_dim})
        return config

# ================= 2. 优化后的卫星决策环境 =================

class SatelliteEnvOptimized:
    """
    优化后的环境：包含避障、功率控制和切换代价
    """
    def __init__(self, num_channels=10):
        self.num_channels = num_channels
        self.last_action = None

    def step(self, action, true_occupancy, predicted_risk):
        """
        action: 选择的信道索引 (0-9)
        true_occupancy: 真实干扰情况 (0/1)
        predicted_risk: 预测模型输出的该信道风险概率 (0.0-1.0)
        """
        # 1. 检测碰撞
        collision = (true_occupancy[action] == 1.0)
        
        # 2. 动态奖励函数设计
        reward = 0
        
        if not collision:
            reward += 2.0  # 基础成功奖励
            # 功率优化：如果预测风险极低且避障成功，给予额外奖励（模拟低功率节省能耗）
            if predicted_risk < 0.2:
                reward += 0.5 
        else:
            reward -= 5.0  # 碰撞惩罚（权重加大，保证安全性第一）

        # 3. 切换代价：避免信道频繁抖动
        if self.last_action is not None and action != self.last_action:
            reward -= 0.2
            
        self.last_action = action
        return reward, collision

# ================= 3. 优化后的 DQN Agent (带目标网络) =================

class DQNAgentOptimized:
    def __init__(self, state_size=10, action_size=10):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=4000)
        self.gamma = 0.95    # 奖励折扣因子
        self.epsilon = 1.0   # 探索率
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.99
        self.learning_rate = 0.0005
        
        # 建立主网络与目标网络（提升训练稳定性）
        self.model = self._build_model()
        self.target_model = self._build_model()
        self.update_target_model()

    def _build_model(self):
        model = Sequential([
            Dense(128, input_dim=self.state_size, activation='relu'),
            Dropout(0.1),
            Dense(128, activation='relu'),
            Dense(self.action_size, activation='linear')
        ])
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate))
        return model

    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        act_values = self.model.predict(state, verbose=0)
        return np.argmax(act_values[0])

    def replay(self, batch_size):
        if len(self.memory) < batch_size: return
        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                # 使用目标网络预测 Q 值
                target = (reward + self.gamma * np.max(self.target_model.predict(next_state, verbose=0)[0]))
            target_f = self.model.predict(state, verbose=0)
            target_f[0][action] = target
            self.model.fit(state, target_f, epochs=1, verbose=0)
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# ================= 4. 闭环系统总控流程 =================

def main_optimized_pipeline():
    # 路径定义
    DATA_PATH = "/root/autodl-tmp/validate/0218/Prediction/situation_data_enhanced"
    PRED_MODEL_PATH = "/root/autodl-tmp/validate/0218/Prediction/Transformer/models/best_hybrid_situation.h5"

    # 1. 数据与模型准备
    print("🛰️ 加载系统组件...")
    X_test = np.load(f"{DATA_PATH}/X_test.npy")
    y_test = np.load(f"{DATA_PATH}/y_test.npy")
    
    custom_objects = {'TransformerBlock': TransformerBlock}
    pred_model = load_model(PRED_MODEL_PATH, custom_objects=custom_objects, compile=False)
    
    env = SatelliteEnvOptimized(num_channels=10)
    agent = DQNAgentOptimized(state_size=10, action_size=10)
    
    batch_size = 32
    episodes = 200 # 训练+评估轮数
    collision_count = 0
    
    print("\n🔥 启动预测驱动的智能抗干扰决策闭环训练...")
    
    for e in range(episodes):
        idx = random.randint(0, len(X_test) - 1)
        state_input = X_test[idx:idx+1]
        
        # A. 预测层：获取未来态势概率
        pred_probs = pred_model.predict(state_input, verbose=0)
        
        # B. 决策层：执行动作
        action = agent.act(pred_probs)
        
        # C. 环境层：计算奖励与真实碰撞
        true_occ = y_test[idx]
        risk_at_action = pred_probs[0][action]
        reward, collision = env.step(action, true_occ, risk_at_action)
        
        if collision: collision_count += 1
        
        # D. 学习过程
        # 为简化演示，这里假设 next_state 为预测出的概率分布
        next_idx = (idx + 1) % len(X_test)
        next_pred = pred_model.predict(X_test[next_idx:next_idx+1], verbose=0)
        agent.remember(pred_probs, action, reward, next_pred, False)
        
        if len(agent.memory) > batch_size:
            agent.replay(batch_size)
            
        if e % 10 == 0:
            agent.update_target_model() # 定期同步目标网络
            success_rate = (1 - collision_count/(e+1)) * 100
            print(f"📡 轮次: {e:3d}/{episodes} | 避障率: {success_rate:6.2f}% | 决策信道: {action} | Epsilon: {agent.epsilon:.2f}")

    print("\n" + "="*45)
    print(f"✅ 系统最终评估指标汇报")
    print(f"🏆 最终平均避障成功率: {(1 - collision_count/episodes)*100:.2f}%")
    print(f"💡 该决策系统已成功串联 MBF-PLENet 感知输出与 Transformer 态势预判")
    print("="*45)

if __name__ == "__main__":
    main_optimized_pipeline()

2026-02-24 14:29:42.169811: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 14:29:42.230978: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 14:29:43.224408: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


🛰️ 加载系统组件...


2026-02-24 14:29:44.708535: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22182 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:27:00.0, compute capability: 8.9



🔥 启动预测驱动的智能抗干扰决策闭环训练...


2026-02-24 14:29:46.158695: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:424] Loaded cuDNN version 8600
2026-02-24 14:29:46.417007: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:637] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


📡 轮次:   0/200 | 避障率: 100.00% | 决策信道: 6 | Epsilon: 1.00
📡 轮次:  10/200 | 避障率:  72.73% | 决策信道: 2 | Epsilon: 1.00
📡 轮次:  20/200 | 避障率:  71.43% | 决策信道: 9 | Epsilon: 1.00
📡 轮次:  30/200 | 避障率:  70.97% | 决策信道: 4 | Epsilon: 1.00


2026-02-24 14:29:52.369384: I tensorflow/compiler/xla/service/service.cc:169] XLA service 0x5587ca984340 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-24 14:29:52.369410: I tensorflow/compiler/xla/service/service.cc:177]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-02-24 14:29:52.374098: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-24 14:29:52.534755: I ./tensorflow/compiler/jit/device_compiler.h:180] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


📡 轮次:  40/200 | 避障率:  75.61% | 决策信道: 2 | Epsilon: 0.91
📡 轮次:  50/200 | 避障率:  76.47% | 决策信道: 7 | Epsilon: 0.83
📡 轮次:  60/200 | 避障率:  77.05% | 决策信道: 9 | Epsilon: 0.75
📡 轮次:  70/200 | 避障率:  80.28% | 决策信道: 4 | Epsilon: 0.68
📡 轮次:  80/200 | 避障率:  80.25% | 决策信道: 5 | Epsilon: 0.61
📡 轮次:  90/200 | 避障率:  81.32% | 决策信道: 5 | Epsilon: 0.55
📡 轮次: 100/200 | 避障率:  83.17% | 决策信道: 7 | Epsilon: 0.50
📡 轮次: 110/200 | 避障率:  80.18% | 决策信道: 7 | Epsilon: 0.45
📡 轮次: 120/200 | 避障率:  80.99% | 决策信道: 0 | Epsilon: 0.41
📡 轮次: 130/200 | 避障率:  82.44% | 决策信道: 6 | Epsilon: 0.37
📡 轮次: 140/200 | 避障率:  82.27% | 决策信道: 2 | Epsilon: 0.33
📡 轮次: 150/200 | 避障率:  82.78% | 决策信道: 0 | Epsilon: 0.30
📡 轮次: 160/200 | 避障率:  83.85% | 决策信道: 0 | Epsilon: 0.27
📡 轮次: 170/200 | 避障率:  84.80% | 决策信道: 8 | Epsilon: 0.25
📡 轮次: 180/200 | 避障率:  85.64% | 决策信道: 0 | Epsilon: 0.22
📡 轮次: 190/200 | 避障率:  85.86% | 决策信道: 0 | Epsilon: 0.20

✅ 系统最终评估指标汇报
🏆 最终平均避障成功率: 86.50%
💡 该决策系统已成功串联 MBF-PLENet 感知输出与 Transformer 态势预判
